<a href="https://colab.research.google.com/github/MurphyKlein/CS4782_final_project/blob/main/notebook/fine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
# Force remount to fix 'Transport endpoint is not connected' errors
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

BASE_DIR = "/content/drive/MyDrive/DL_Final_Project"

# ─────────────────────────────────────────────
# 1. WEATHER DATASET  (mpi_roof_2025.csv)
# ─────────────────────────────────────────────
WEATHER_DIR  = f"{BASE_DIR}/weather_data"
WEATHER_FILE = os.path.join(WEATHER_DIR, "mpi_roof_2025.csv")

def load_weather(filepath):
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return None
    df = pd.read_csv(filepath, encoding='unicode_escape')
    print(f"[Weather] Raw shape: {df.shape}")

    # drop any datetime / string columns
    date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
    if date_cols:
        df = df.drop(columns=date_cols)

    df = df.select_dtypes(include=[np.number])

    # Keep all weather data (zeros allowed)
    df = df.dropna()
    print(f"[Weather] Final shape: {df.shape}")
    return df

weather_df = load_weather(WEATHER_FILE)

# ─────────────────────────────────────────────
# 2. ELECTRICITY DATASET  (LD2011_2014.txt)
# ─────────────────────────────────────────────
ELECTRICITY_DIR  = f"{BASE_DIR}/electricity_data"
ELECTRICITY_FILE = os.path.join(ELECTRICITY_DIR, "LD2011_2014.txt")

def load_electricity(filepath):
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return None
    print("[Electricity] Loading... (this may take a moment)")
    # Added low_memory=False to handle mixed types and potentially avoid parser crashes
    df = pd.read_csv(filepath, sep=';', decimal=',', index_col=0, parse_dates=True, low_memory=False)
    print(f"[Electricity] Raw shape: {df.shape}")

    # resample 15-min → hourly
    df = df.resample('1h').mean()

    # Drop columns that contain any 0s (for Electricity only)
    cols_with_zeros = df.columns[(df == 0).any()]
    print(f"[Electricity] Dropping {len(cols_with_zeros)} columns containing zeros.")
    df = df.drop(columns=cols_with_zeros)

    df = df.dropna()
    print(f"[Electricity] Final shape: {df.shape}")
    return df

electricity_df = load_electricity(ELECTRICITY_FILE)

# ─────────────────────────────────────────────
# 3. SPLIT + SCALE
# ─────────────────────────────────────────────
def split_and_scale(df, train_ratio=0.7, val_ratio=0.1):
    if df is None: return None, None, None, None
    n       = len(df)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    train = df.iloc[:n_train].values.astype(np.float32)
    val   = df.iloc[n_train : n_train + n_val].values.astype(np.float32)
    test  = df.iloc[n_train + n_val:].values.astype(np.float32)

    scaler = StandardScaler().fit(train)
    return scaler.transform(train), scaler.transform(val), scaler.transform(test), scaler

if weather_df is not None:
    print("\n── Weather splits ──")
    w_train, w_val, w_test, w_scaler = split_and_scale(weather_df)
    print(f"  train {w_train.shape}  val {w_val.shape}  test {w_test.shape}")

if electricity_df is not None:
    print("\n── Electricity splits ──")
    e_train, e_val, e_test, e_scaler = split_and_scale(electricity_df)
    print(f"  train {e_train.shape}  val {e_val.shape}  test {e_test.shape}")

print("\nAll done — data ready for DataLoader.")

[Weather] Raw shape: (52560, 22)
[Weather] Final shape: (52560, 21)
[Electricity] Loading... (this may take a moment)
[Electricity] Raw shape: (140256, 370)
[Electricity] Dropping 280 columns containing zeros.
[Electricity] Final shape: (35065, 90)

── Weather splits ──
  train (36792, 21)  val (5256, 21)  test (10512, 21)

── Electricity splits ──
  train (24545, 90)  val (3506, 90)  test (7014, 90)

All done — data ready for DataLoader.


In [ ]:
print("--- Weather Dataset Preview ---")
display(weather_df.head())

print("\n--- Electricity Dataset Preview ---")
display(electricity_df.head())

--- Weather Dataset Preview ---


,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),...,wv (m/s),max. wv (m/s),wd (deg),rain (mm),raining (s),SWDR (W/m²),PAR (µmol/m²/s),max. PAR (µmol/m²/s),Tlog (degC),CO2 (ppm)
0,997.30,-1.56,271.80,-2.97,90.0,5.45,4.90,0.54,3.06,4.92,...,1.61,3.87,148.9,0.0,0.0,0.0,0.0,0.0,6.24,456.9
1,997.05,-0.90,272.48,-2.66,87.8,5.72,5.02,0.70,3.14,5.03,...,1.71,3.05,145.6,0.0,0.0,0.0,0.0,0.0,6.32,451.7
2,996.96,-0.51,272.88,-2.44,86.7,5.89,5.10,0.78,3.19,5.12,...,1.61,3.31,146.9,0.0,0.0,0.0,0.0,0.0,6.46,451.0
3,996.77,-0.27,273.14,-2.26,86.3,5.99,5.17,0.82,3.23,5.19,...,1.86,2.93,137.0,0.0,0.0,0.0,0.0,0.0,6.63,450.4
4,996.62,-0.09,273.33,-2.14,86.0,6.07,5.22,0.85,3.26,5.24,...,1.25,2.23,131.3,0.0,0.0,0.0,0.0,0.0,6.82,450.4



--- Electricity Dataset Preview ---


,MT_156,MT_158,MT_159,MT_166,MT_168,MT_169,MT_171,MT_172,MT_176,MT_180,...,MT_318,MT_319,MT_320,MT_321,MT_323,MT_324,MT_326,MT_327,MT_329,MT_330
2011-01-01 00:00:00,68.203369,38.787879,20.363985,831.135531,56.023326,22.983670,146.988922,80.792860,41.150912,93.291732,...,92.685753,47.336090,60.661464,56.506239,987.520325,240.633803,138.395677,270.557342,130.117647,45.708333
2011-01-01 01:00:00,68.359326,38.673128,19.887452,769.752747,59.224041,19.736311,151.540785,76.634496,41.482587,86.271451,...,98.742331,47.188423,59.035674,59.054622,982.408537,230.792254,127.892768,285.032154,133.305147,44.612500
2011-01-01 02:00:00,68.359326,39.010695,21.087165,783.516484,59.503386,19.590778,143.202417,75.650685,38.243781,85.881435,...,96.947853,46.743946,56.665273,61.923224,939.695122,225.510563,130.701372,272.966238,126.316176,46.332812
2011-01-01 03:00:00,68.203369,39.004011,20.129310,778.021978,58.941874,19.736311,140.158610,74.034869,40.485075,83.541342,...,93.624744,45.118133,57.083333,59.054622,957.987805,226.390845,131.954489,264.927653,127.047794,45.865625
2011-01-01 04:00:00,68.359326,38.338904,21.087165,808.269231,58.659707,19.159942,140.929003,76.637609,39.987562,89.781591,...,97.972904,44.527466,54.990245,62.307105,967.134146,225.070423,132.883416,260.096463,128.150735,43.675000


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 1: Imports & device
# ─────────────────────────────────────────────────────────────────
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2: Dataset
# ─────────────────────────────────────────────────────────────────
class TimeSeriesDataset(Dataset):
    """
    Sliding window dataset.
    x : (seq_len,  C)  — look-back window
    y : (pred_len, C)  — forecast target
    """
    def __init__(self, data: np.ndarray, seq_len: int, pred_len: int):
        self.data     = torch.tensor(data, dtype=torch.float32)
        self.seq_len  = seq_len
        self.pred_len = pred_len

    def __len__(self):
        return len(self.data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        x = self.data[idx            : idx + self.seq_len]
        y = self.data[idx+self.seq_len : idx + self.seq_len + self.pred_len]
        return x, y


def make_loaders(train, val, test, seq_len, pred_len, batch_size=32):
    def _loader(arr, shuffle):
        ds = TimeSeriesDataset(arr, seq_len, pred_len)
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                          num_workers=2, pin_memory=True, drop_last=False)
    return _loader(train, True), _loader(val, False), _loader(test, False)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3: PatchTST model
# ─────────────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    def __init__(self, patch_len: int, stride: int, d_model: int, seq_len: int):
        super().__init__()
        self.patch_len = patch_len
        self.stride    = stride
        # paper formula: floor((L - P) / S) + 2
        self.n_patches = math.floor((seq_len - patch_len) / stride) + 2
        self.proj      = nn.Linear(patch_len, d_model)
        self.pos_emb   = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward(self, x):
        # x : (B*C, seq_len)
        # pad end with last value (paper's strategy)
        x   = torch.cat([x, x[:, -1:].expand(-1, self.stride)], dim=1)
        # unfold → (B*C, n_patches, patch_len)
        x   = x.unfold(dimension=1, size=self.patch_len, step=self.stride)
        x   = self.proj(x) + self.pos_emb[:, :x.size(1), :]
        return x  # (B*C, N, d_model)


class PatchTST(nn.Module):
    def __init__(
        self,
        n_channels : int,
        seq_len    : int,
        pred_len   : int,
        patch_len  : int   = 12,
        stride     : int   = 12,
        d_model    : int   = 128,
        n_heads    : int   = 16,
        n_layers   : int   = 3,
        d_ff       : int   = 256,
        dropout    : float = 0.2,
    ):
        super().__init__()
        self.n_channels = n_channels
        self.pred_len   = pred_len
        self.patch_len  = patch_len
        self.stride     = stride

        self.patch_embed = PatchEmbedding(patch_len, stride, d_model, seq_len)
        N = self.patch_embed.n_patches
        self.N       = N
        self.d_model = d_model

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = n_heads,
            dim_feedforward = d_ff,
            dropout         = dropout,
            batch_first     = True,
            norm_first      = False,
        )
        self.transformer  = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm         = nn.BatchNorm1d(d_model)

        # supervised forecast head
        self.forecast_head = nn.Linear(N * d_model, pred_len)

        # masked autoencoder head (pretraining only)
        self.pretrain_head = nn.Linear(d_model, patch_len)

    # ── instance normalisation (paper sec 3.1) ──────────────────
    @staticmethod
    def _inst_norm(x):
        # x : (B, L, C)
        mean = x.mean(dim=1, keepdim=True)
        std  = x.std(dim=1, keepdim=True).clamp(min=1e-5)
        return (x - mean) / std, mean, std

    @staticmethod
    def _inst_denorm(x, mean, std):
        return x * std + mean

    # ── shared channel-independent backbone ─────────────────────
    def _encode(self, x):
        """x : (B, L, C)  →  z : (B*C, N, d_model)"""
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L)   # (B*C, L)
        z = self.patch_embed(x)                      # (B*C, N, d_model)
        z = self.transformer(z)                      # (B*C, N, d_model)
        # BatchNorm over d_model
        z = self.norm(z.reshape(-1, self.d_model)).reshape(B * C, self.N, self.d_model)
        return z, B, C

    # ── supervised forecast ──────────────────────────────────────
    def forecast(self, x):
        x, mean, std = self._inst_norm(x)
        z, B, C = self._encode(x)                   # (B*C, N, d_model)
        z   = z.reshape(B * C, -1)                  # (B*C, N*d_model)
        out = self.forecast_head(z)                  # (B*C, pred_len)
        out = out.reshape(B, C, self.pred_len).permute(0, 2, 1)  # (B, pred_len, C)
        # denormalise using per-series stats broadcast over pred_len
        out = self._inst_denorm(out, mean, std)
        return out

    # ── masked pretraining ───────────────────────────────────────
    def pretrain_step(self, x, mask_ratio: float = 0.4):
        x, _, _ = self._inst_norm(x)
        B, L, C = x.shape
        BC      = B * C
        xflat   = x.permute(0, 2, 1).reshape(BC, L)   # (B*C, L)

        # raw patches (before positional embedding)
        xpad    = torch.cat([xflat, xflat[:, -1:].expand(-1, self.stride)], dim=1)
        patches = xpad.unfold(1, self.patch_len, self.stride)  # (B*C, N, P)
        N       = patches.size(1)

        # random mask (1 = masked)
        n_mask  = int(N * mask_ratio)
        noise   = torch.rand(BC, N, device=x.device)
        ids     = noise.argsort(dim=1)
        mask    = torch.zeros(BC, N, device=x.device)
        mask.scatter_(1, ids[:, :n_mask], 1.0)

        # project patches + zero masked + add pos emb
        proj_patches = self.patch_embed.proj(patches)           # (B*C, N, d_model)
        pos          = self.patch_embed.pos_emb[:, :N, :]
        inp          = proj_patches * (1 - mask.unsqueeze(-1)) + pos

        z    = self.transformer(inp)
        z    = self.norm(z.reshape(-1, self.d_model)).reshape(BC, N, self.d_model)
        recon = self.pretrain_head(z)                           # (B*C, N, P)

        # MSE loss on masked patches only
        loss = ((recon - patches) ** 2 * mask.unsqueeze(-1)).sum() \
             / (mask.sum() * self.patch_len + 1e-8)
        return loss

    def forward(self, x, mode='forecast'):
        if mode == 'forecast':
            return self.forecast(x)
        return self.pretrain_step(x)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 4: Training helpers
# ─────────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, mode='forecast'):
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        if mode == 'forecast':
            loss = F.mse_loss(model(x), y)
        else:
            loss = model(x, mode='pretrain')
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    mse = mae = n = 0
    for x, y in loader:
        x, y  = x.to(DEVICE), y.to(DEVICE)
        pred  = model(x)
        mse  += F.mse_loss(pred, y, reduction='sum').item()
        mae  += F.l1_loss (pred, y, reduction='sum').item()
        n    += y.numel()
    return mse / n, mae / n


def fit(model, tr_loader, val_loader, epochs, lr, mode='forecast', tag=''):
    opt  = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sch  = CosineAnnealingLR(opt, T_max=epochs)
    best_mse, best_weights = float('inf'), None

    for ep in range(1, epochs + 1):
        tr_loss = train_epoch(model, tr_loader, opt, mode)
        sch.step()

        if mode == 'forecast':
            val_mse, val_mae = evaluate(model, val_loader)
            if val_mse < best_mse:
                best_mse     = val_mse
                best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if ep % 5 == 0 or ep == 1:
                print(f"  [{tag}] ep {ep:3d}/{epochs} | "
                      f"train={tr_loss:.4f} | val_MSE={val_mse:.4f} | val_MAE={val_mae:.4f}")
        else:
            if ep % 20 == 0 or ep == 1:
                print(f"  [{tag}] ep {ep:3d}/{epochs} | pretrain_loss={tr_loss:.5f}")

    # restore best checkpoint
    if best_weights:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_weights.items()})
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 5: Full linear-probing experiment
#
# Exactly matches Table 4 pipeline:
#   Step 1 — pretrain         100 epochs  (masked autoencoder)
#   Step 2 — linear probing    20 epochs  (backbone frozen)
#   Step 3 — end-to-end FT     20 epochs  (full model, low lr)
# ─────────────────────────────────────────────────────────────────
def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = ('forecast_head' in name)

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True


def run_experiment(
    train_data, val_data, test_data,
    n_channels,
    pred_len,
    dataset_name,
    # architecture (paper defaults for large datasets)
    seq_len   = 512,
    patch_len = 12,
    stride    = 12,
    d_model   = 128,
    n_heads   = 16,
    n_layers  = 3,
    d_ff      = 256,
    dropout   = 0.2,
    # training
    batch_size      = 32,
    pretrain_epochs = 100,
    linprobe_epochs = 20,
    finetune_epochs = 20,
    pretrain_lr     = 1e-4,
    linprobe_lr     = 1e-4,
    finetune_lr     = 1e-5,
):
    print(f"\n{'='*60}")
    print(f"  {dataset_name}  |  pred_len={pred_len}  |  C={n_channels}")
    print(f"{'='*60}")

    tr_loader, val_loader, te_loader = make_loaders(
        train_data, val_data, test_data,
        seq_len=seq_len, pred_len=pred_len, batch_size=batch_size
    )

    model = PatchTST(
        n_channels=n_channels, seq_len=seq_len, pred_len=pred_len,
        patch_len=patch_len, stride=stride,
        d_model=d_model, n_heads=n_heads, n_layers=n_layers,
        d_ff=d_ff, dropout=dropout,
    ).to(DEVICE)
    print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Patches per series: {model.N}")

    # ── step 1: pretrain ────────────────────────────────────────
    print(f"\n[1/3] Pretraining  ({pretrain_epochs} epochs, mask_ratio=0.4)")
    fit(model, tr_loader, val_loader,
        epochs=pretrain_epochs, lr=pretrain_lr,
        mode='pretrain', tag='Pre')

    # ── step 2: linear probing ───────────────────────────────────
    print(f"\n[2/3] Linear probing  ({linprobe_epochs} epochs, backbone frozen)")
    freeze_backbone(model)
    fit(model, tr_loader, val_loader,
        epochs=linprobe_epochs, lr=linprobe_lr,
        mode='forecast', tag='LinProbe')

    # ── step 3: fine-tuning ──────────────────────────────────────
    print(f"\n[3/3] Fine-tuning  ({finetune_epochs} epochs, full model)")
    unfreeze_all(model)
    fit(model, tr_loader, val_loader,
        epochs=finetune_epochs, lr=finetune_lr,
        mode='forecast', tag='FineTune')

    # ── test ─────────────────────────────────────────────────────
    test_mse, test_mae = evaluate(model, te_loader)
    print(f"\n  ► Test  MSE={test_mse:.4f}  MAE={test_mae:.4f}")
    return model, test_mse, test_mae

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 6: Run all horizons — Weather & Electricity
# T ∈ {96, 192, 336, 720}  matching Table 4
# ─────────────────────────────────────────────────────────────────
PRED_LENS = [96, 192, 336, 720]
results   = {}

# ── Weather ──────────────────────────────────────────────────────
for T in PRED_LENS:
    _, mse, mae = run_experiment(
        train_data   = w_train,
        val_data     = w_val,
        test_data    = w_test,
        n_channels   = w_train.shape[1],   # 21
        pred_len     = T,
        dataset_name = 'Weather',
    )
    results[('Weather', T)] = (round(mse, 4), round(mae, 4))

# ── Electricity ───────────────────────────────────────────────────
for T in PRED_LENS:
    _, mse, mae = run_experiment(
        train_data   = e_train,
        val_data     = e_val,
        test_data    = e_test,
        n_channels   = e_train.shape[1],   # 90 after zero-col removal
        pred_len     = T,
        dataset_name = 'Electricity',
    )
    results[('Electricity', T)] = (round(mse, 4), round(mae, 4))

# ── summary table ─────────────────────────────────────────────────
print("\n" + "="*56)
print(f"{'Dataset':<14} {'T':>5}  {'MSE':>8}  {'MAE':>8}  {'Paper MSE':>10}  {'Paper MAE':>10}")
print("="*56)

paper = {   # Table 4 fine-tuning reference numbers
    ('Weather',     96): (0.144, 0.193),
    ('Weather',    192): (0.190, 0.236),
    ('Weather',    336): (0.244, 0.280),
    ('Weather',    720): (0.320, 0.335),
    ('Electricity', 96): (0.126, 0.221),
    ('Electricity',192): (0.145, 0.238),
    ('Electricity',336): (0.164, 0.256),
    ('Electricity',720): (0.193, 0.291),
}

for (ds, T), (mse, mae) in results.items():
    pmse, pmae = paper.get((ds, T), ('—', '—'))
    print(f"{ds:<14} {T:>5}  {mse:>8.4f}  {mae:>8.4f}  {str(pmse):>10}  {str(pmae):>10}")
print("="*56)


  Weather  |  pred_len=96  |  C=21
  Params: 934,892
  Patches per series: 43

[1/3] Pretraining  (100 epochs, mask_ratio=0.4)
  [Pre] ep   1/100 | pretrain_loss=0.95634
  [Pre] ep  20/100 | pretrain_loss=0.20580
  [Pre] ep  40/100 | pretrain_loss=0.19885
  [Pre] ep  60/100 | pretrain_loss=0.19644
  [Pre] ep  80/100 | pretrain_loss=0.19514
  [Pre] ep 100/100 | pretrain_loss=0.19498

[2/3] Linear probing  (20 epochs, backbone frozen)
  [LinProbe] ep   1/20 | train=0.3850 | val_MSE=0.2996 | val_MAE=0.3176
  [LinProbe] ep   5/20 | train=0.3616 | val_MSE=0.2939 | val_MAE=0.3131
  [LinProbe] ep  10/20 | train=0.3601 | val_MSE=0.2929 | val_MAE=0.3120
  [LinProbe] ep  15/20 | train=0.3592 | val_MSE=0.2925 | val_MAE=0.3119
  [LinProbe] ep  20/20 | train=0.3588 | val_MSE=0.2925 | val_MAE=0.3117

[3/3] Fine-tuning  (20 epochs, full model)
  [FineTune] ep   1/20 | train=0.3499 | val_MSE=0.2840 | val_MAE=0.3041
  [FineTune] ep   5/20 | train=0.3323 | val_MSE=0.2770 | val_MAE=0.2980
  [FineTune] e

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL: Table 5 — Transfer Learning (Fine-tuning)
#
# Pipeline:
#   1. Pretrain backbone on Electricity
#   2. Swap forecast head for Weather channels
#   3. Linear probe 10 epochs (backbone frozen)   ← paper uses
#   4. Fine-tune full model 20 epochs             ← two-step strategy
#   5. Evaluate on Weather test set
# ─────────────────────────────────────────────────────────────────

def run_transfer_finetune(
    # source (pretrain)
    src_train, src_val,
    src_channels,
    # target
    tgt_train, tgt_val, tgt_test,
    tgt_channels,
    pred_len,
    # architecture
    seq_len         = 512,
    patch_len       = 12,
    stride          = 12,
    d_model         = 128,
    n_heads         = 16,
    n_layers        = 3,
    d_ff            = 256,
    dropout         = 0.2,
    # training
    batch_size      = 32,
    pretrain_epochs = 100,
    linprobe_epochs = 10,    # paper: 10 epochs before full fine-tune
    finetune_epochs = 20,
    pretrain_lr     = 1e-4,
    linprobe_lr     = 1e-4,
    finetune_lr     = 1e-5,
    src_name        = 'Electricity',
    tgt_name        = 'Weather',
):
    print(f"\n{'='*60}")
    print(f"  Transfer Fine-tune | pretrain={src_name} → target={tgt_name}")
    print(f"  pred_len={pred_len} | src_C={src_channels} | tgt_C={tgt_channels}")
    print(f"{'='*60}")

    # ── loaders ──────────────────────────────────────────────────
    src_tr_loader, src_val_loader, _ = make_loaders(
        src_train, src_val, src_val,
        seq_len=seq_len, pred_len=pred_len, batch_size=batch_size
    )
    tgt_tr_loader, tgt_val_loader, tgt_te_loader = make_loaders(
        tgt_train, tgt_val, tgt_test,
        seq_len=seq_len, pred_len=pred_len, batch_size=batch_size
    )

    # ── build model on source channel count ──────────────────────
    model = PatchTST(
        n_channels = src_channels,
        seq_len    = seq_len,
        pred_len   = pred_len,
        patch_len  = patch_len,
        stride     = stride,
        d_model    = d_model,
        n_heads    = n_heads,
        n_layers   = n_layers,
        d_ff       = d_ff,
        dropout    = dropout,
    ).to(DEVICE)
    print(f"  Pretrain model params: {sum(p.numel() for p in model.parameters()):,}")

    # ── step 1: pretrain on Electricity ──────────────────────────
    print(f"\n[1/3] Pretraining on {src_name}  ({pretrain_epochs} epochs)")
    fit(model, src_tr_loader, src_val_loader,
        epochs=pretrain_epochs, lr=pretrain_lr,
        mode='pretrain', tag='Pre')

    # ── swap forecast head for target ────────────────────────────
    N = model.N
    model.n_channels    = tgt_channels
    model.forecast_head = nn.Linear(N * d_model, pred_len).to(DEVICE)
    print(f"\n  Swapped forecast head → ({N * d_model}, {pred_len})")

    # ── step 2: linear probe first (warm up head) ────────────────
    print(f"\n[2/3] Linear probing on {tgt_name}  ({linprobe_epochs} epochs, backbone frozen)")
    freeze_backbone(model)
    fit(model, tgt_tr_loader, tgt_val_loader,
        epochs=linprobe_epochs, lr=linprobe_lr,
        mode='forecast', tag='LinProbe')

    # ── step 3: full fine-tuning ─────────────────────────────────
    print(f"\n[3/3] Fine-tuning on {tgt_name}  ({finetune_epochs} epochs, full model)")
    unfreeze_all(model)
    fit(model, tgt_tr_loader, tgt_val_loader,
        epochs=finetune_epochs, lr=finetune_lr,
        mode='forecast', tag='FineTune')

    # ── test ─────────────────────────────────────────────────────
    test_mse, test_mae = evaluate(model, tgt_te_loader)
    print(f"\n  ► Test  MSE={test_mse:.4f}  MAE={test_mae:.4f}")
    return model, test_mse, test_mae

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL: Run Table 5 — Fine-tuning for all 4 horizons
# ─────────────────────────────────────────────────────────────────
PRED_LENS = [96, 192, 336, 720]
transfer_finetune_results = {}

for T in PRED_LENS:
    _, mse, mae = run_transfer_finetune(
        src_train    = e_train,
        src_val      = e_val,
        src_channels = e_train.shape[1],   # 90

        tgt_train    = w_train,
        tgt_val      = w_val,
        tgt_test     = w_test,
        tgt_channels = w_train.shape[1],   # 21

        pred_len     = T,
    )
    transfer_finetune_results[T] = (round(mse, 4), round(mae, 4))

# ── combined summary: lin probe + fine-tune vs Table 5 ───────────
paper_table5 = {
    #  T :  (finetune_MSE, finetune_MAE, linprobe_MSE, linprobe_MAE)
     96:  (0.145, 0.195, 0.163, 0.216),
    192:  (0.193, 0.243, 0.205, 0.252),
    336:  (0.244, 0.280, 0.253, 0.289),
    720:  (0.321, 0.337, 0.320, 0.336),
}

print("\n" + "="*80)
print(f"  Table 5 — Transfer Learning (Electricity → Weather)")
print("="*80)
print(f"{'T':>5}  "
      f"{'FT MSE':>8}  {'FT MAE':>8}  {'Paper FT MSE':>13}  {'Paper FT MAE':>13}  "
      f"{'LP MSE':>8}  {'LP MAE':>8}")
print("-"*80)
for T in PRED_LENS:
    ft_mse,  ft_mae  = transfer_finetune_results[T]
    lp_mse,  lp_mae  = transfer_linprobe_results.get(T, ('—', '—'))
    pft_mse, pft_mae, _, _ = paper_table5[T]
    print(f"{T:>5}  "
          f"{ft_mse:>8.4f}  {ft_mae:>8.4f}  {pft_mse:>13.3f}  {pft_mae:>13.3f}  "
          f"{str(lp_mse):>8}  {str(lp_mae):>8}")
print("="*80)